# 00 - Master Revision Notebook
### ML and AML Sem VII - All 25 Questions, 9 Families

**How to use:** Skim top-to-bottom in 10 minutes before the exam. Each section has the exact model line, eval line, and plot code you need. Copy any snippet during the exam.

---

## Table of Contents
1. [All Imports](#all-imports)
2. [Common Patterns](#common-patterns)
3. [Decision Trees (Q1-Q4)](#decision-trees)
4. [Cross Validation (Q5)](#cross-validation)
5. [Naive Bayes (Q6)](#naive-bayes)
6. [SVM (Q7-Q9)](#svm)
7. [Regression (Q10-Q11)](#regression)
8. [K-Means and Elbow (Q12-Q14)](#kmeans)
9. [DBSCAN (Q15-Q16)](#dbscan)
10. [PCA (Q17-Q21)](#pca)
11. [LDA (Q22-Q25)](#lda)
12. [Quick Comparison Table](#comparison)
13. [Pitfalls Checklist](#pitfalls)


In [1]:
# <a id='all-imports'></a>
# ALL IMPORTS - every library used across all 9 families
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for sandbox

# Core numeric / data
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn - datasets
from sklearn.datasets import (
    load_iris, load_wine, load_breast_cancer, load_digits,
    make_blobs, make_moons, make_classification
)

# sklearn - preprocessing and split
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# sklearn - supervised models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# sklearn - unsupervised models
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# sklearn - metrics
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, silhouette_score,
    mean_squared_error, r2_score
)
from matplotlib.colors import ListedColormap

print('All imports OK')


/home/devpanchal/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-s0yr2w7f because there was an issue with the default path (/home/devpanchal/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


All imports OK


In [2]:
# <a id='common-patterns'></a>
# COMMON PATTERNS - reusable skeletons
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Load CSV dataset
# df = pd.read_csv('datasets/diabetes.csv')
# df = pd.read_csv('datasets/Social_Network_Ads.csv')
# df = pd.read_csv('datasets/Salary.csv')

# 2. Load sklearn dataset
# from sklearn.datasets import load_iris
# data = load_iris(); X, y = data.data, data.target

# 3. Inspect data
# print(df.shape); print(df.head(3))
# print(df.columns.tolist()); print(df.dtypes)
# print(df.isnull().sum())

# 4. Encode categorical column
# df['Gender'] = LabelEncoder().fit_transform(df['Gender'])

# 5. Train / test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scale features (fit on train ONLY)
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test  = scaler.transform(X_test)

# 7. Fit and predict
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# 8. Accuracy and confusion matrix
# print('Accuracy:', accuracy_score(y_test, y_pred))
# cm = confusion_matrix(y_test, y_pred)
# ConfusionMatrixDisplay(cm).plot(); plt.show()

# 9. 2D scatter colored by label/cluster
def scatter2d(X, labels, title='Scatter', xlabel='Feature 1', ylabel='Feature 2'):
    unique = np.unique(labels)
    for lbl in unique:
        mask = labels == lbl
        plt.scatter(X[mask, 0], X[mask, 1], label=str(lbl), s=30)
    plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.legend(); plt.tight_layout(); plt.show()

print('Common patterns loaded')


Common patterns loaded


## Decision Trees (Q1-Q4) <a id='decision-trees'></a>

**Covers:** Q1 (diabetes, max_depth=3, full train, plot_tree), Q2 (iris: DT vs SVM), Q3 (iris: max_leaf_nodes=8 vs min_samples_leaf=5), Q4 (diabetes: DT vs LogisticRegression).

**Key parameters:** `max_depth` limits levels | `max_leaf_nodes` caps leaves | `min_samples_leaf` min samples per leaf | `criterion='gini'` default.

**Pitfall:** Q1 trains on FULL dataset (no split). Q4 says 'Logistic Classifier' - use LogisticRegression.


In [3]:
# Decision Trees - Q1: diabetes, full train, max_depth=3
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree

df = pd.read_csv('datasets/diabetes.csv')
X = df.drop('Outcome', axis=1)  # all features except label
y = df['Outcome']

# Q1: train on complete dataset (question says so - no split)
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)
print('Q1 training accuracy:', round(model.score(X, y), 4))

plt.figure(figsize=(18, 7))
plot_tree(model, feature_names=list(X.columns),
          class_names=['No Diabetes', 'Diabetes'], filled=True, rounded=True)
plt.title('Q1 - Decision Tree (diabetes, max_depth=3)')
plt.tight_layout(); plt.show()


Q1 training accuracy: 0.776


In [4]:
# Q2: iris - DT vs SVM | Q4: diabetes - DT vs LogisticRegression
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Q2 - Iris
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
dt_acc  = accuracy_score(y_test, DecisionTreeClassifier(random_state=42).fit(X_train, y_train).predict(X_test))
svm_acc = accuracy_score(y_test, SVC(random_state=42).fit(X_train, y_train).predict(X_test))
print(f'Q2 - Decision Tree: {dt_acc:.4f}  |  SVM: {svm_acc:.4f}')

# Q4 - Diabetes  (Logistic Classifier = LogisticRegression)
df = pd.read_csv('datasets/diabetes.csv')
X4 = df.drop('Outcome', axis=1); y4 = df['Outcome']
X4_tr, X4_te, y4_tr, y4_te = train_test_split(X4, y4, test_size=0.2, random_state=42)
dt4_acc = accuracy_score(y4_te, DecisionTreeClassifier(random_state=42).fit(X4_tr, y4_tr).predict(X4_te))
lr4_acc = accuracy_score(y4_te, LogisticRegression(max_iter=1000, random_state=42).fit(X4_tr, y4_tr).predict(X4_te))
print(f'Q4 - Decision Tree: {dt4_acc:.4f}  |  Logistic Reg: {lr4_acc:.4f}')

# Bar charts side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, models, accs, title in [
    (axes[0], ['DT', 'SVM'], [dt_acc, svm_acc], 'Q2 - Iris: DT vs SVM'),
    (axes[1], ['DT', 'Logistic'], [dt4_acc, lr4_acc], 'Q4 - Diabetes: DT vs Logistic')
]:
    bars = ax.bar(models, accs, color=['steelblue', 'tomato'], edgecolor='black')
    for b, a in zip(bars, accs):
        ax.text(b.get_x() + b.get_width()/2, b.get_height()+0.01, f'{a:.4f}', ha='center')
    ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy'); ax.set_title(title)
plt.tight_layout(); plt.show()


Q2 - Decision Tree: 1.0000  |  SVM: 1.0000


Q4 - Decision Tree: 0.7468  |  Logistic Reg: 0.7468


In [5]:
# Q3 - iris: max_leaf_nodes=8 vs min_samples_leaf=5
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

m_leaf = DecisionTreeClassifier(max_leaf_nodes=8, random_state=42).fit(X_train, y_train)
m_min  = DecisionTreeClassifier(min_samples_leaf=5, random_state=42).fit(X_train, y_train)
acc_leaf = accuracy_score(y_test, m_leaf.predict(X_test))
acc_min  = accuracy_score(y_test, m_min.predict(X_test))
print(f'Q3 max_leaf_nodes=8  : {acc_leaf:.4f}')
print(f'Q3 min_samples_leaf=5: {acc_min:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
plot_tree(m_leaf, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, ax=axes[0])
axes[0].set_title(f'max_leaf_nodes=8  | Acc={acc_leaf:.4f}')
plot_tree(m_min, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, ax=axes[1])
axes[1].set_title(f'min_samples_leaf=5 | Acc={acc_min:.4f}')
plt.tight_layout(); plt.show()

# VIVA NOTES:
# max_depth=3: 3 decision levels; deeper = more complex = may overfit
# max_leaf_nodes caps leaves; min_samples_leaf prevents tiny leaves - both are pruning
# DT vs SVM: DT is interpretable (plot it); SVM is a black box but often more accurate
# 'Logistic Classifier' in Q4 = LogisticRegression - uses sigmoid for probability
# Accuracy = correct predictions / total predictions
# How to read plot_tree: each node shows split feature, gini score, samples, class distribution


Q3 max_leaf_nodes=8  : 1.0000
Q3 min_samples_leaf=5: 1.0000


## Cross Validation (Q5) <a id='cross-validation'></a>

**Covers:** Q5 - LOOCV on 100-sample, 5-feature binary synthetic dataset with DecisionTreeClassifier.

**Key parameters:** `LeaveOneOut()` - n_splits = n_samples | each fold trains on n-1 samples, tests on 1 sample.

**Pitfall:** LOOCV with 100 samples = 100 iterations. Per-fold accuracy is always 0 or 1 (single test sample). Average is the useful metric.


In [6]:
# Q5 - LOOCV with DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

# Synthetic dataset: 100 samples, 5 features, binary target
X, y = make_classification(n_samples=100, n_features=5, n_classes=2,
                            n_informative=3, random_state=42)
print('Dataset shape:', X.shape, ' | Labels:', np.unique(y))

loo = LeaveOneOut()
accs = []
for fold, (train_idx, test_idx) in enumerate(loo.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]  # test = 1 sample
    y_train, y_test = y[train_idx], y[test_idx]
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    fold_acc = accuracy_score(y_test, model.predict(X_test))  # 0.0 or 1.0
    accs.append(fold_acc)
    if fold < 5 or fold >= 95:  # print only first 5 and last 5
        print(f'Fold {fold+1:3d}: accuracy = {fold_acc:.1f}')

print('Average LOOCV accuracy:', round(float(np.mean(accs)), 4))

# Bar plot: green=correct, red=wrong
plt.figure(figsize=(14, 3))
colors = ['green' if a == 1.0 else 'red' for a in accs]
plt.bar(range(1, 101), accs, color=colors, width=0.8)
plt.xlabel('Fold (sample index)'); plt.ylabel('Accuracy (0 or 1)')
plt.title('Q5 - LOOCV per Fold | Avg=' + str(round(float(np.mean(accs)), 4)))
plt.tight_layout(); plt.show()

# VIVA NOTES:
# LOOCV: n_splits = n_samples (100 folds here); each test set has exactly 1 sample
# Per-fold accuracy is 0.0 or 1.0 - average gives unbiased accuracy estimate
# Expensive for large n; K-Fold (k=5 or 10) is a faster alternative
# LOOCV preferred for small datasets where every sample matters
# Why unbiased: each sample gets tested exactly once across all folds


Dataset shape: (100, 5)  | Labels: [0 1]
Fold   1: accuracy = 1.0
Fold   2: accuracy = 0.0
Fold   3: accuracy = 1.0
Fold   4: accuracy = 1.0
Fold   5: accuracy = 1.0


Fold  96: accuracy = 1.0
Fold  97: accuracy = 1.0
Fold  98: accuracy = 0.0
Fold  99: accuracy = 1.0
Fold 100: accuracy = 1.0
Average LOOCV accuracy: 0.83


## Naive Bayes (Q6) <a id='naive-bayes'></a>

**Covers:** Q6 - GaussianNB on sklearn breast_cancer (569 samples, 30 features, binary labels).

**Key parameters:** GaussianNB assumes Gaussian distribution per feature per class | no key hyperparameters to tune | `var_smoothing=1e-9` default.

**Pitfall:** Use GaussianNB for continuous features. Load from sklearn (not CSV). No scaling required but will not hurt.


In [7]:
# Q6 - GaussianNB on breast cancer (sklearn)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = load_breast_cancer()
X, y = data.data, data.target
print('Shape:', X.shape, '| Classes:', data.target_names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GaussianNB()  # no key hyperparameters for basic use
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=data.target_names))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Q6 - Naive Bayes Confusion Matrix')
plt.tight_layout(); plt.show()

# VIVA NOTES:
# GaussianNB: P(feature|class) assumed Gaussian per feature per class
# 'Naive' = assumes features are independent (rarely true, but works well in practice)
# Why use: fast, no scaling needed, good baseline for high-dimensional continuous data
# Heatmap: diagonal = correct predictions; off-diagonal = errors
# Follow-up: 'Why naive?' - independence assumption simplifies math but rarely holds


Shape: (569, 30) | Classes: ['malignant' 'benign']
Accuracy: 0.9737
              precision    recall  f1-score   support

   malignant       1.00      0.93      0.96        43
      benign       0.96      1.00      0.98        71

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



## SVM (Q7-Q9) <a id='svm'></a>

**Covers:** Q7 (Social_Network_Ads, Age+EstimatedSalary, linear vs RBF, decision boundary), Q8 (breast_cancer 30 features, confusion matrix + PCA 2D viz), Q9 (iris 3 classes, confusion matrix + boundary).

**Key parameters:** `kernel='linear'` or `'rbf'` | `C` = regularization | `gamma` = RBF width | ALWAYS StandardScaler before SVM.

**Pitfall:** Q7 uses only Age and EstimatedSalary (drop User ID and Gender). Q8/Q9 use sklearn loaders. Scale before SVM every time.


In [8]:
# Q7 - SVM linear vs RBF on Social_Network_Ads
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from matplotlib.colors import ListedColormap

df = pd.read_csv('datasets/Social_Network_Ads.csv')
X = df[['Age', 'EstimatedSalary']].values  # only 2 features
y = df['Purchased'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()  # MANDATORY before SVM
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

svc_lin = SVC(kernel='linear', C=1, random_state=42).fit(X_train_sc, y_train)
svc_rbf = SVC(kernel='rbf', C=1, gamma=0.1, random_state=42).fit(X_train_sc, y_train)
print('Q7 Linear SVC accuracy:', round(accuracy_score(y_test, svc_lin.predict(X_test_sc)), 4))
print('Q7 RBF SVC    accuracy:', round(accuracy_score(y_test, svc_rbf.predict(X_test_sc)), 4))

# Decision boundary plot
def plot_boundary(model, X_sc, y, title, ax):
    h = 0.02
    x0_min, x0_max = X_sc[:, 0].min()-1, X_sc[:, 0].max()+1
    x1_min, x1_max = X_sc[:, 1].min()-1, X_sc[:, 1].max()+1
    xx, yy = np.meshgrid(np.arange(x0_min, x0_max, h), np.arange(x1_min, x1_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA', '#AAAAFF']))
    ax.scatter(X_sc[:, 0], X_sc[:, 1], c=y, cmap=ListedColormap(['red', 'blue']),
               edgecolors='k', s=20)
    ax.set_title(title); ax.set_xlabel('Age (scaled)'); ax.set_ylabel('Salary (scaled)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_boundary(svc_lin, X_train_sc, y_train, 'Q7 - Linear SVC', axes[0])
plot_boundary(svc_rbf, X_train_sc, y_train, 'Q7 - RBF SVC (C=1, gamma=0.1)', axes[1])
plt.tight_layout(); plt.show()


Q7 Linear SVC accuracy: 0.85
Q7 RBF SVC    accuracy: 0.9333


In [9]:
# Q8 - SVM on breast_cancer (30 features), confusion matrix + PCA 2D viz
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train); X_test_sc = sc.transform(X_test)

model = SVC(kernel='rbf', random_state=42).fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)
print('Q8 Breast Cancer SVC accuracy:', round(accuracy_score(y_test, y_pred), 4))

# PCA 2D projection to visualize 30D data (for plot only)
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_test_sc)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=y_test, cmap='bwr', s=20)
axes[0].set_title('Q8 - True Labels (PCA 2D)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_pred, cmap='bwr', s=20)
axes[1].set_title('Q8 - Predicted Labels (PCA 2D)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.tight_layout(); plt.show()

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Q8 - SVC Confusion Matrix'); plt.tight_layout(); plt.show()


Q8 Breast Cancer SVC accuracy: 0.9825


In [10]:
# Q9 - SVM on Iris (3 classes), confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train); X_test_sc = sc.transform(X_test)

model = SVC(kernel='rbf', C=1, gamma='scale', random_state=42).fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)
print('Q9 Iris SVC accuracy:', round(accuracy_score(y_test, y_pred), 4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Q9 - Iris SVC Confusion Matrix'); plt.tight_layout(); plt.show()

# VIVA NOTES:
# Kernel trick: maps to high-dim space where data becomes linearly separable
# C (regularization): high C = low bias + high variance; low C = smoother boundary
# gamma (RBF): high = tight boundary per point (may overfit); low = smoother
# Always StandardScaler before SVM - it is distance-based and scale-sensitive
# PCA 2D in Q8 is for visualization only - model trained on all 30 features
# gamma='scale': auto-sets gamma=1/(n_features * X.var()) - safe default


Q9 Iris SVC accuracy: 1.0


## Regression (Q10-Q11) <a id='regression'></a>

**Covers:** Q10 (LogisticRegression on 10-row telecom churn data), Q11 (LinearRegression on Salary.csv 'Years of Experience' vs 'Salary', full dataset).

**Key parameters:** `LogisticRegression(max_iter=1000)` | `LinearRegression()` no key params | `r2_score` 1.0=perfect.

**Pitfall:** Q10 has 10 rows - test set is 2 samples, accuracy is coarse. Q11 uses Salary.csv with `.dropna()` (12 missing rows). Train on FULL dataset.


In [11]:
# Q10 - Logistic Regression on 10-row churn dataset
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

data = {
    'Age':             [25, 34, 45, 29, 50, 38, 42, 35, 48, 55],
    'Account_Length':  [12, 24, 36, 18, 48, 30, 42, 24, 36, 60],
    'Monthly_Charges': [70, 90, 80, 100, 60, 80, 90, 70, 80, 100],
    'Churn':           [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)
X = df.drop('Churn', axis=1); y = df['Churn']

# 80/20 split -> only 2 test samples, so accuracy is coarse
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', len(X_train), '| Test:', len(X_test))

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Predicted:', y_pred, '| Actual:', y_test.values)
print('Q10 Accuracy:', round(accuracy_score(y_test, y_pred), 4))
# Note: Only 10 rows - test set is 2 samples, accuracy is coarse


Train: 8 | Test: 2
Predicted: [1 1] | Actual: [0 1]
Q10 Accuracy: 0.5


In [12]:
# Q11 - Linear Regression on Salary.csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Load only needed columns, drop NaN (12 missing rows in Salary.csv)
df = pd.read_csv('datasets/Salary.csv')[['Years of Experience', 'Salary']].dropna()
print('Shape after dropna:', df.shape)

X = df[['Years of Experience']].values  # 2D for sklearn
y = df['Salary'].values

# Train on COMPLETE dataset (question says so)
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
print('R2 Score :', round(r2_score(y, y_pred), 4))
print('MSE      :', round(mean_squared_error(y, y_pred), 2))

# Plot 1: data + fitted line
plt.figure(figsize=(6, 4))
plt.scatter(X, y, color='steelblue', s=20, label='Actual data')
plt.plot(X, y_pred, color='red', linewidth=2, label='Fitted line')
plt.xlabel('Years of Experience'); plt.ylabel('Salary')
plt.title('Q11 - Linear Regression Fit'); plt.legend(); plt.tight_layout(); plt.show()

# Plot 2: residual plot (scatter around y=0 means good fit)
residuals = y - y_pred
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, color='darkorange', s=20)
plt.axhline(0, color='black', linewidth=1, linestyle='--')
plt.xlabel('Predicted Salary'); plt.ylabel('Residuals')
plt.title('Q11 - Residual Plot'); plt.tight_layout(); plt.show()

# VIVA NOTES:
# Logistic Regression: sigmoid output = probability; threshold 0.5 for class decision
# Linear Regression: minimizes MSE to find best-fit line y = mx + b
# R2=1.0: perfect fit; R2=0: model no better than mean; R2<0: worse than mean
# Residual plot: random scatter around y=0 = good model; pattern = model is wrong
# Q10 coarse accuracy: 2 test samples - each wrong answer moves accuracy by 50%


Shape after dropna: (373, 2)
R2 Score : 0.8655
MSE      : 312090043.8


## K-Means and Elbow (Q12-Q14) <a id='kmeans'></a>

**Covers:** Q12 (elbow on 14 random points), Q13 (make_blobs 300 samples, subplot grid k=2 to 7), Q14 (35 random points, k=3 + varying k impact).

**Key parameters:** `n_clusters=k` | `inertia_` = within-cluster SSE | `cluster_centers_` | `n_init=10` default.

**Pitfall:** Inertia always decreases with k - use elbow or silhouette to choose optimal k. Need to pre-specify k (unlike DBSCAN).


In [13]:
# Q12 - Elbow Method on 14 generated points
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

np.random.seed(42)
X = np.random.rand(14, 2) * 100  # exactly 14 points, scale 0-100

inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Number of Clusters (k)'); plt.ylabel('Inertia (WCSS)')
plt.title('Q12 - Elbow Method (14 points)')
plt.xticks(list(k_range)); plt.tight_layout(); plt.show()
print('Look for the bend (elbow) in the curve - that is the optimal k')


Look for the bend (elbow) in the curve - that is the optimal k


In [14]:
# Q13 - make_blobs 300 samples, subplot grid k=2 to 7
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

X, _ = make_blobs(n_samples=300, centers=4, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for i, k in enumerate(range(2, 8)):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    axes[i].scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=15)
    axes[i].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                    c='black', marker='X', s=100, label='Centroids')
    axes[i].set_title(f'k={k}')
    axes[i].legend(loc='upper right', fontsize=7)
plt.suptitle('Q13 - KMeans k=2 to 7 (make_blobs 300 samples)')
plt.tight_layout(); plt.show()

# Conclusion:
# k=4 best matches natural groupings (data generated with 4 centers).
# k < 4: clusters too broad, dissimilar points merged (underfitting).
# k > 4: natural groups split arbitrarily; inertia decreases but no real gain.


In [15]:
# Q14 - 35 random points, k=3 default, then varying k
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

np.random.seed(42)
X = np.random.rand(35, 2) * 100  # exactly 35 points

# Default k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=km3.labels_, cmap='tab10', s=40, edgecolors='k')
plt.scatter(km3.cluster_centers_[:, 0], km3.cluster_centers_[:, 1],
            c='black', marker='X', s=150, label='Centroids')
plt.title('Q14 - KMeans k=3 (35 random points)'); plt.legend()
plt.tight_layout(); plt.show()

# Varying k=2,3,4,5
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, k in zip(axes, [2, 3, 4, 5]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    ax.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='tab10', s=30, edgecolors='k')
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=100)
    ax.set_title(f'k={k} | inertia={km.inertia_:.0f}')
plt.suptitle('Q14 - Varying k Impact (35 random points)')
plt.tight_layout(); plt.show()

# Conclusion:
# Too few clusters: dissimilar points forced into same group.
# Too many clusters: natural groups split arbitrarily; inertia still decreases.
# Use elbow plot or silhouette score to find where adding k stops being useful.

# VIVA NOTES:
# Inertia (WCSS): sum of squared distances of each point to its centroid
# Elbow: k where adding one more cluster gives little inertia reduction
# KMeans needs k pre-specified; DBSCAN finds clusters automatically
# n_init=10: runs 10 times with different random inits, keeps best result
# Silhouette score: -1 to 1; higher = better separated clusters


## DBSCAN (Q15-Q16) <a id='dbscan'></a>

**Covers:** Q15 (make_moons, eps=0.2, min_samples=5), Q16 (random 5-col data 0-100, cols 3 and 4, eps=5, minPts=5).

**Key parameters:** `eps` = neighborhood radius | `min_samples` = minPts in question | `labels_==-1` = noise.

**Pitfall:** eps depends on data scale - 0-100 data needs eps ~5, 0-1 data needs eps ~0.1. Plot noise points separately in black.


In [16]:
# Q15 - DBSCAN on make_moons
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

model = DBSCAN(eps=0.2, min_samples=5)
labels = model.fit_predict(X)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int(np.sum(labels == -1))
print(f'Q15 - Clusters: {n_clusters}  |  Noise points: {n_noise}')

plt.figure(figsize=(7, 4))
unique_labels = sorted(set(labels))
colors = plt.cm.tab10(np.linspace(0, 1, max(max(unique_labels)+1, 1)))
for lbl in unique_labels:
    mask = labels == lbl
    color = 'black' if lbl == -1 else colors[lbl]
    marker = 'x' if lbl == -1 else 'o'
    name = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    plt.scatter(X[mask, 0], X[mask, 1], c=[color], marker=marker, s=20, label=name)
plt.title('Q15 - DBSCAN on make_moons (eps=0.2, min_samples=5)')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.legend(); plt.tight_layout(); plt.show()


Q15 - Clusters: 2  |  Noise points: 0


In [17]:
# Q16 - DBSCAN on columns 3 and 4 of 5-column random data, eps=5, minPts=5
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# Scale 0-100 so that eps=5 is meaningful (~5% of range)
np.random.seed(42)
data = np.random.rand(200, 5) * 100
X = data[:, 2:4]  # 3rd and 4th columns (0-indexed 2 and 3)
print('Q16 - Using cols 3 and 4, shape:', X.shape)

model = DBSCAN(eps=5, min_samples=5)  # minPts=5 -> min_samples=5 in sklearn
labels = model.fit_predict(X)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int(np.sum(labels == -1))
print(f'Clusters: {n_clusters}  |  Noise: {n_noise}')

plt.figure(figsize=(6, 5))
unique_labels = sorted(set(labels))
for lbl in unique_labels:
    mask = labels == lbl
    color = 'black' if lbl == -1 else plt.cm.tab10(lbl / max(max(unique_labels), 1))
    name  = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    plt.scatter(X[mask, 0], X[mask, 1], c=[color], s=15, label=name)
plt.title('Q16 - DBSCAN (cols 3 and 4, eps=5, min_samples=5)')
plt.xlabel('Column 3'); plt.ylabel('Column 4')
plt.legend(loc='upper right', fontsize=7); plt.tight_layout(); plt.show()

# VIVA NOTES:
# DBSCAN does NOT need k pre-specified - finds clusters automatically
# Core point: >= min_samples neighbors within eps; Border: within eps of core
# Noise (label=-1): not reachable from any core point
# eps too small: everything is noise | eps too large: everything merges
# Advantage over KMeans: handles arbitrary shapes (crescents, rings), detects outliers
# Q16: data scaled 0-100 so eps=5 captures ~5% neighborhood - forms meaningful clusters


Q16 - Using cols 3 and 4, shape: (200, 2)
Clusters: 5  |  Noise: 166


## PCA (Q17-Q21) <a id='pca'></a>

**Covers:** Q17/18/21 (Iris 4D -> 2D scatter), Q19 (100 random samples 3 features -> 2D), Q20 (Wine all components, explained variance chart).

**Key parameters:** `n_components=2` | `explained_variance_ratio_` | `fit_transform(X)` - no y needed (unsupervised).

**Pitfall:** StandardScaler BEFORE PCA (Q17 requires it explicitly). PCA is unsupervised - does not use labels.


In [18]:
# Q17/18/21 - PCA on Iris: 4D -> 2D scatter
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

iris = load_iris()
X, y = iris.data, iris.target
print('Original shape:', X.shape)  # 150 x 4

# Q17: standardize before PCA
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)  # 150 x 2
print('PCA shape:', X_pca.shape)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(4))
print('Total variance explained:', round(float(pca.explained_variance_ratio_.sum()), 4))

# 2D scatter (same plot for Q17, Q18, Q21)
colors = ['red', 'green', 'blue']
plt.figure(figsize=(7, 5))
for i, species in enumerate(iris.target_names):
    mask = y == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i],
                label=species, s=30, edgecolors='k', linewidths=0.3)
plt.xlabel('PC1 (Principal Component 1)')
plt.ylabel('PC2 (Principal Component 2)')
plt.title('Q17/18/21 - PCA on Iris (4D -> 2D)')
plt.legend(); plt.tight_layout(); plt.show()


Original shape: (150, 4)
PCA shape: (150, 2)
Explained variance ratio: [0.7296 0.2285]
Total variance explained: 0.9581


In [19]:
# Q19 - PCA on 100 random samples with 3 features
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)
X = np.random.randn(100, 3)  # exactly 100 samples, 3 features
print('Q19 Before PCA:', X.shape)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
print('Q19 After PCA :', X_pca.shape)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(4))

plt.figure(figsize=(5, 4))
plt.scatter(X_pca[:, 0], X_pca[:, 1], color='steelblue', s=30,
            edgecolors='k', linewidths=0.3)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('Q19 - PCA on 100 random samples (3 -> 2D)')
plt.tight_layout(); plt.show()

# Q20 - PCA on Wine, all components, variance chart
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine = load_wine()
X_w = StandardScaler().fit_transform(wine.data)  # 13 features
pca_all = PCA().fit(X_w)  # all components
evr = pca_all.explained_variance_ratio_

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(range(1, len(evr)+1), evr, color='steelblue', label='Per-component')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(range(1, len(evr)+1), np.cumsum(evr), color='red', marker='o',
         linewidth=2, label='Cumulative')
ax2.set_ylabel('Cumulative Explained Variance', color='red')
ax2.axhline(0.95, color='red', linestyle='--', linewidth=0.8)
ax1.set_title('Q20 - PCA Explained Variance (Wine, 13 features)')
plt.tight_layout(); plt.show()

# VIVA NOTES:
# PCA: finds orthogonal directions (PCs) of maximum variance in data
# Standardize BEFORE PCA - features with large scale dominate otherwise
# explained_variance_ratio_[0]: fraction of total variance in PC1
# Cumulative: first 2-3 PCs typically capture 70-95% for structured data
# PCA is unsupervised - does NOT use labels (unlike LDA)
# Scree plot bar chart: look for elbow to choose n_components


Q19 Before PCA: (100, 3)
Q19 After PCA : (100, 2)
Explained variance ratio: [0.4538 0.3221]


## LDA (Q22-Q25) <a id='lda'></a>

**Covers:** Q22 (Wine 3 classes -> 2D), Q23 (breast_cancer 2 classes -> 1D histogram), Q24 (digits 10 classes -> 2D), Q25 (Iris 3 classes -> 2D).

**Key parameters:** `n_components` max = `min(n_classes-1, n_features)` | `fit_transform(X, y)` - LDA needs labels (supervised).

**Pitfall:** Always pass y to fit_transform. Q23 breast_cancer has 2 classes -> max 1 component -> 1D histogram plot.


In [20]:
# Q22 - LDA on Wine (3 classes -> 2D) | Q25 - LDA on Iris (3 classes -> 2D)
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_iris
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['red', 'green', 'blue']

# Q22 - Wine: 3 classes, 13 features -> max = min(3-1, 13) = 2
wine = load_wine(); X_w, y_w = wine.data, wine.target
X_lda_w = LinearDiscriminantAnalysis(n_components=2).fit_transform(X_w, y_w)
for i, cls in enumerate(wine.target_names):
    mask = y_w == i
    axes[0].scatter(X_lda_w[mask, 0], X_lda_w[mask, 1], c=colors[i], label=cls, s=25)
axes[0].set_title('Q22 - LDA Wine (3 classes -> 2D)')
axes[0].set_xlabel('LD1'); axes[0].set_ylabel('LD2'); axes[0].legend()

# Q25 - Iris: 3 classes, 4 features -> max = min(3-1, 4) = 2
iris = load_iris(); X_i, y_i = iris.data, iris.target
X_lda_i = LinearDiscriminantAnalysis(n_components=2).fit_transform(X_i, y_i)
for i, cls in enumerate(iris.target_names):
    mask = y_i == i
    axes[1].scatter(X_lda_i[mask, 0], X_lda_i[mask, 1], c=colors[i], label=cls, s=25)
axes[1].set_title('Q25 - LDA Iris (3 classes -> 2D)')
axes[1].set_xlabel('LD1'); axes[1].set_ylabel('LD2'); axes[1].legend()

plt.tight_layout(); plt.show()


/tmp/ipykernel_22/2233278884.py:6: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 2, figsize=(13, 5))


In [21]:
# Q23 - LDA on breast_cancer (2 classes -> 1D histogram)
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

data = load_breast_cancer()
X, y = data.data, data.target
# 2 classes -> max n_components = min(2-1, 30) = 1
lda = LinearDiscriminantAnalysis(n_components=1)
X_lda = lda.fit_transform(X, y)  # shape: (569, 1)
print('Q23 LDA output shape:', X_lda.shape)

plt.figure(figsize=(7, 4))
for i, cls in enumerate(data.target_names):
    plt.hist(X_lda[y == i, 0], bins=30, alpha=0.6, label=cls)
plt.xlabel('LD1 (Linear Discriminant 1)'); plt.ylabel('Count')
plt.title('Q23 - LDA Breast Cancer (2 classes -> 1D)')
plt.legend(); plt.tight_layout(); plt.show()


Q23 LDA output shape: (569, 1)


In [22]:
# Q24 - LDA on digits (10 classes -> 2D)
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

digits = load_digits()
X, y = digits.data, digits.target
# 10 classes, 64 features -> max = min(10-1, 64) = 9; use 2 for 2D plot
X_lda = LinearDiscriminantAnalysis(n_components=2).fit_transform(X, y)
print('Q24 Digits LDA shape:', X_lda.shape)

plt.figure(figsize=(9, 6))
sc = plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='tab10', s=15, alpha=0.7)
cbar = plt.colorbar(sc, ticks=range(10))
cbar.set_label('Digit')
plt.title('Q24 - LDA on Digits (10 classes -> 2D)')
plt.xlabel('LD1'); plt.ylabel('LD2')
plt.tight_layout(); plt.show()

# VIVA NOTES:
# LDA: maximizes ratio of between-class variance to within-class variance
# LDA is SUPERVISED - must pass y to fit_transform (unlike PCA)
# n_components limit: min(n_classes-1, n_features) - hard mathematical constraint
# Q23: 2 classes -> max 1 component -> plot 1D as histogram
# LDA vs PCA: LDA separates classes best; PCA captures most variance
# LDA better than PCA when class separation is the goal


Q24 Digits LDA shape: (1797, 2)


## Quick Comparison Table <a id='comparison'></a>

| Method | Type | Needs Scaling | Key Parameter | When to Use |
|---|---|---|---|---|
| Decision Tree | Supervised | No | `max_depth`, `max_leaf_nodes` | Interpretable, non-linear |
| SVM | Supervised | **YES** | `kernel`, `C`, `gamma` | High-dim, small-medium data |
| Logistic Regression | Supervised | Recommended | `max_iter` | Binary/multi-class probability |
| Naive Bayes | Supervised | No | `var_smoothing` | Continuous features, fast baseline |
| KMeans | Unsupervised | Recommended | `n_clusters` | Spherical clusters, k pre-specified |
| DBSCAN | Unsupervised | Scale-aware | `eps`, `min_samples` | Arbitrary shapes, finds noise |
| PCA | Unsupervised (dim reduce) | **YES** | `n_components` | Visualize, reduce before model |
| LDA | Supervised (dim reduce) | Optional | `n_components` (max=classes-1) | Maximize class separation |

**PCA vs LDA:** PCA unsupervised (max variance) vs LDA supervised (max class separation). LDA max components = n_classes - 1.

**KMeans vs DBSCAN:** KMeans needs k, spherical only, outliers distort centroids. DBSCAN auto-finds k, handles arbitrary shapes, marks outliers as -1.


In [23]:
# <a id='pitfalls'></a>
# PITFALLS CHECKLIST - read before every question

# 1. Scale before SVM and PCA:
#    scaler = StandardScaler()
#    X_train = scaler.fit_transform(X_train)  # fit on train ONLY
#    X_test  = scaler.transform(X_test)        # same scaler, transform only

# 2. LDA n_components limit:
#    n_components <= min(n_classes - 1, n_features)
#    breast_cancer (2 classes) -> max 1  | iris/wine (3) -> max 2  | digits (10) -> max 9

# 3. Encode categorical before modelling:
#    df['Gender'] = LabelEncoder().fit_transform(df['Gender'])
#    (Q7 uses only Age + EstimatedSalary - drop Gender and User ID)

# 4. Always random_state=42:
#    train_test_split(..., random_state=42)
#    KMeans(..., random_state=42) | DecisionTreeClassifier(random_state=42)

# 5. Clustering plots need 2D data:
#    Use first 2 features or PCA(n_components=2) before plotting

# 6. DBSCAN eps depends on data scale:
#    Data 0-1:   eps ~0.1-0.5
#    Data 0-100: eps ~5-15  (Q16 uses eps=5 on 0-100 data)

# 7. Tiny test set accuracy is coarse:
#    Q10: 10 rows -> 2 test samples -> each wrong answer moves accuracy by 50%

# 8. LOOCV has exactly 1 test sample per fold:
#    Per-fold accuracy = 0.0 or 1.0
#    Average across all folds is the useful metric

# 9. Q1 and Q11 train on FULL dataset (question says so):
#    model.fit(X, y)  # no split

# 10. Salary.csv has missing values - always dropna:
#     df = pd.read_csv('datasets/Salary.csv')[['Years of Experience','Salary']].dropna()

print('Pitfalls loaded - you are exam ready!')


Pitfalls loaded - you are exam ready!
